# Week 8 - 비지도학습 실습(고급): 서울시 따릉이 대여소 공간 군집화 (실제 데이터)

## 🚲 서울시설공단 따릉이 운영팀의 고민

당신은 **서울시설공단 따릉이 운영팀의 데이터 사이언티스트**다. 서울 전역에 3,000개 이상의 따릉이 대여소가 운영 중이며, 매일 **자전거 재배치 트럭**이 수요-공급 불균형을 해소한다. 대여소별 위치와 월별 대여 건수 데이터를 활용해 두 가지 **서로 다른** 운영 의사결정을 지원해야 한다.

| # | 해결하고자 하는 문제/질문 | 적합 알고리즘 | 이유 |
|---|---|---|---|
| **Q1** | 재배치 트럭 **관리 구역을 몇 개로 나눠** 각 구역당 한 대씩 배치할까? 각 트럭의 주 거점은 어디에? | **K-Means** | 공간을 균등하게 분할, centroid = 트럭 차고지 후보 |
| **Q2** | 시민들이 자연스럽게 만들어내는 **수요 핫스팟(상권·교통허브)** 은 어디이고, 이용량이 적어 운영 효율이 낮은 **저이용 대여소**는 어디인가? | **DBSCAN** | 밀도 기반 자연 클러스터 + 노이즈(저이용 대여소) 식별 |

## 📊 데이터셋
- **출처**: 서울 열린데이터광장 (data.seoul.go.kr) 에서 다운로드한 2개 파일을 merge 하여 본 실습을 위한 하나의 csv 파일로 가공함
  - [따릉이 대여소 마스터 정보](https://data.seoul.go.kr/dataList/OA-21235/S/1/datasetView.do) (위도/경도)
  - [대여소별 월별 이용정보](https://data.seoul.go.kr/dataList/OA-15249/F/1/datasetView.do) (월 대여/반납 건수)
- **기간**: 2025년 7~11월 (5개월)
- **규모**: 1,760개 대여소 (25개 자치구 전역)
- **주요 컬럼**: `station_id`, `lat`, `lon`, `monthly_usage`, `gu`, `rent_202507`~`rent_202511`

## 🎯 학습 목표
1. **대여소 단위 데이터**(point data with attribute)에 군집화를 적용
2. 지리 데이터에 DBSCAN을 올바르게 적용시 사용하는 **Haversine 거리**(실제 지구/지표면상에서의 포인트간 거리 계산에 사용) 활용
3. 실제 서울 데이터에서 발생하는 **데이터 품질 이슈**(결측, 이상값 등)에 대한 대응/고민
4. 하이퍼파라미터 선택 및 평가

---

---
## 1. 환경 설정 및 데이터 로딩


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors

import folium
from folium.plugins import HeatMap

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

Colab 환경에서 한글 폰트 설정을 위한 라이브러리 설치
(한글이 깨져서 나타날 경우 설치)

In [ ]:
!pip install koreanize-matplotlib

import koreanize_matplotlib

In [ ]:
# ==== 데이터 로딩 ====

url = "https://raw.githubusercontent.com/agtechresearch/MLapplications-graduate/refs/heads/main/dataset/SeoulBike_dataset.csv"
df = pd.read_csv(url)

In [ ]:
print(f'로드 완료: {len(df):,}개 대여소')
print(f'컬럼: {list(df.columns)}')
df.head()

## 2. 탐색적 데이터 분석 (EDA)

실제 데이터이므로 먼저 데이터 특성을 이해한다.

In [ ]:
print(f'총 대여소 수: {len(df):,}')
print(f'자치구 수: {df["gu"].nunique()}')
print(f'위도 범위: [{df.lat.min():.4f}, {df.lat.max():.4f}]')
print(f'경도 범위: [{df.lon.min():.4f}, {df.lon.max():.4f}]')

print(f'\n월 평균 대여량 통계:')
print(df['monthly_usage'].describe().round(0))

print(f'\n월 평균 대여량 TOP 10 대여소:')
print(df[['station_name', 'gu', 'monthly_usage']].head(10).to_string(index=False))

대여소 분포, 이용량 분포, 자치구별 대여소 수: 시각화

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sc = axes[0].scatter(df['lon'], df['lat'],
                     s=df['monthly_usage']/100,
                     c=df['monthly_usage'],
                     cmap='YlOrRd', alpha=0.6,
                     edgecolor='black', linewidth=0.15)
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
axes[0].set_title(f'대여소 분포 (n={len(df):,})\n크기/색 = 월 평균 대여량')
plt.colorbar(sc, ax=axes[0], label='Monthly Usage')

axes[1].hist(df['monthly_usage'], bins=60, edgecolor='black')
axes[1].set_xlabel('Monthly Usage'); axes[1].set_ylabel('# of Stations')
axes[1].set_title('이용량 분포 (Long-tail)')
axes[1].set_yscale('log')
axes[1].axvline(df['monthly_usage'].median(), color='red', linestyle='--',
                label=f'median={df["monthly_usage"].median():.0f}')
axes[1].legend()

gu_stats = df.groupby('gu').agg(n=('station_id', 'count')).sort_values('n', ascending=True)
axes[2].barh(gu_stats.index, gu_stats['n'])
axes[2].set_xlabel('대여소 수'); axes[2].set_title('자치구별 대여소 수')
axes[2].tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.show()

** 데이터 관찰 및 인사이트 **
- **Long-tail 분포**: 대부분 월 1,000건 수준이지만 상위 몇 개는 10,000건 이상 (예: 마곡나루역, 잠실 롯데월드타워)
- **자치구별 편차**: 송파·서초·강남·강서·영등포가 상위권

## 3. [Part 1] K-Means로 재배치 트럭 관리 구역 설계 (Q1)

> **해결하고자 하는 문제/질문**: 재배치 트럭 $k$대를 어디에 배치할까?

- **K-Means**: 모든 대여소를 동등하게 → 공간적으로 균등한 구역

In [ ]:
# Elbow Method: 몇 대의 트럭이 필요한가? (트럭 1대가 담당 가능한 규모를 함께 고려해 결정해야 함)

coords = df[['lat', 'lon']].values
weights = df['monthly_usage'].values

ks = range(2, 16)
inertias = []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(coords)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(list(ks), inertias, 'o-', linewidth=2, markersize=8)
plt.axvline(x=8, color='red', linestyle='--', alpha=0.5, label='후보 k=8')
plt.xlabel('k (트럭 대수)'); plt.ylabel('Inertia')
plt.title('Elbow Method — 몇 대의 재배치 트럭이 필요한가?')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

print('💡 실제 데이터에서는 elbow가 뚜렷하지 않음 — 서울이 전반적으로 연속 분포이기 때문.')
print('   운영 관점에서 "트럭 1대가 담당 가능한 규모"를 함께 고려해 결정해야 함.')

In [ ]:
# k=8로 K-Means 클러스터링
K = 8

model_kmeans = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE)
model_kmeans.fit(coords)

# 각 대여소의 클러스터 레이블과 클러스터 센터(트럭 배치 위치) 계산
labels_kmeans = model_kmeans.labels_
centers_kmeans = model_kmeans.cluster_centers_

In [ ]:
# 클러스터링 결과 시각화
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

ax.scatter(
    df['lon'], df['lat'],
    s=df['monthly_usage'] / 120,
    c=labels_kmeans,
    cmap='tab10',
    alpha=0.5,
    edgecolor='black',
    linewidth=0.1
 )

ax.scatter(centers_kmeans[:, 1], centers_kmeans[:, 0],
           c='red', marker='X', s=400, edgecolor='white', linewidth=2, zorder=5)

centers_df = pd.DataFrame(centers_kmeans, columns=['lat', 'lon'])
centers_df['label'] = 'T' + centers_df.index.astype(str)
centers_df.apply(
    lambda r: ax.annotate(r['label'], (r['lon'], r['lat']),
                          fontsize=11, fontweight='bold',
                          xytext=(8, 8), textcoords='offset points'),
    axis=1
 )

ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('K-Means\n모든 대여소를 동등하게')

plt.tight_layout()
plt.show()

### 💡 K-Means 결과 해석
- 공간 중심(geometric center)에 centroid → 외곽 지역까지 균등 포함
  - 차고지 목적이 "면적 커버"라면 가장 적절히 균등 클러스터링
  - 하지만, 만약 차고지 목적이 "대여량에 대한 대응 속도"라면 일반 K-Means 보다는 Weighted K-Means 를 통한 접근이 더 적합할 수 있음

## 3. [Part 2] DBSCAN으로 수요 핫스팟 탐지 (Q2)

> **해결하고자 하는 문제/질문**: 시민들이 자연스럽게 형성한 수요 군집은 어디인가? 저이용 대여소는?

### 3-1. Haversine 거리 설정

지리 데이터는 구면 위에 있으므로 **haversine 거리**(실제 지표면 거리)를 사용함

In [ ]:
# K-Means는 유클리드 거리를 최소화하는 방식으로 클러스터링을 수행하기 때문에
# 지리적 좌표(위도/경도)를 그대로 사용할 경우 실제 지구 표면에서의 거리가 왜곡될 수 있음.
# 특히 서울처럼 위도/경도 범위가 작은 지역에서는 왜곡이 크지 않지만,
# 더 넓은 지역에서는 클러스터링 결과가 실제 지리적 분포와 다르게 나올 수 있음.
EARTH_RADIUS_KM = 6371.0088
coords_rad = np.radians(coords)

def km_to_rad(km):
    return km / EARTH_RADIUS_KM

### 3-2. k-distance graph로 eps 선택

In [ ]:
MIN_SAMPLES = 10

nn = NearestNeighbors(n_neighbors=MIN_SAMPLES, metric='haversine').fit(coords_rad)
dists, _ = nn.kneighbors(coords_rad)
k_dists_m = np.sort(dists[:, -1]) * EARTH_RADIUS_KM * 1000

plt.figure(figsize=(8, 4))
plt.plot(k_dists_m, linewidth=2)
plt.axhline(y=800, color='red', linestyle='--', alpha=0.6, label='eps = 800m')
plt.xlabel('points (sorted)')
plt.ylabel(f'{MIN_SAMPLES}-th nearest neighbor distance (m)')
plt.title(f'k-distance graph (min_samples={MIN_SAMPLES})')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

print(f'\n💡 서울은 대여소 밀도가 비교적 균질해서 elbow가 부드러움.')
print(f'   중앙값 {np.median(k_dists_m):.0f}m, 80% 지점 {np.percentile(k_dists_m, 80):.0f}m')
print(f'   → 약 800m를 eps로 선택')

### 3-3. DBSCAN

In [ ]:
EPS_M = 800
eps_rad = km_to_rad(EPS_M / 1000)

model_dbscan = DBSCAN(eps=eps_rad, min_samples=MIN_SAMPLES, metric='haversine')
model_dbscan.fit(coords_rad)

labels_dbscan = model_dbscan.labels_

def summarize(labels, name):
    n_c = len(set(labels)) - (1 if -1 in labels else 0)
    n_n = (labels == -1).sum()
    print(f'{name:30s}: {n_c}개 핫스팟, 노이즈 {n_n}개 ({n_n/len(labels)*100:.1f}%)')

summarize(labels_dbscan, 'DBSCAN')

In [ ]:
# DBSCAN 클러스터링 결과 시각화
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

mn = labels_dbscan == -1
ax.scatter(df.loc[mn, 'lon'], df.loc[mn, 'lat'],
           s=15, alpha=0.4, color='lightgray', marker='x',
           label=f'저이용/외곽 ({mn.sum()})')

uc = sorted(set(labels_dbscan) - {-1})
colors = plt.cm.turbo(np.linspace(0.05, 0.95, max(len(uc), 1)))
for c, color in zip(uc, colors):
    m = labels_dbscan == c
    ax.scatter(df.loc[m, 'lon'], df.loc[m, 'lat'],
               s=df.loc[m, 'monthly_usage']/100, alpha=0.7, color=color,
               edgecolor='black', linewidth=0.1)

ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title(f'DBSCAN (eps={EPS_M}m)')
ax.legend(loc='upper left')

plt.tight_layout()
plt.show()

### 💡 DBSCAN 결과 해석
- 공간 밀도 고려 → 자치구 경계에 가까운 자연스러운 핫스팟

## 4. 시각화를 통한 두 알고리즘 직접 비교

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

ax = axes[0]
for k in range(K):
    m = labels_kmeans == k
    ax.scatter(df.loc[m, 'lon'], df.loc[m, 'lat'],
               s=df.loc[m, 'monthly_usage']/120, alpha=0.5)
ax.scatter(centers_kmeans[:, 1], centers_kmeans[:, 0],
           c='red', marker='X', s=400, edgecolor='white', linewidth=2, zorder=5)
ax.set_title(f'K-Means (k={K})\n→ Q1: 재배치 트럭 {K}대의 차고지 + 관리 구역')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')

ax = axes[1]
mn = labels_dbscan == -1
ax.scatter(df.loc[mn, 'lon'], df.loc[mn, 'lat'],
           s=15, alpha=0.4, color='lightgray', marker='x')
uc = sorted(set(labels_dbscan) - {-1})
colors = plt.cm.turbo(np.linspace(0.05, 0.95, max(len(uc), 1)))
for c, color in zip(uc, colors):
    m = labels_dbscan == c
    ax.scatter(df.loc[m, 'lon'], df.loc[m, 'lat'],
               s=df.loc[m, 'monthly_usage']/100, alpha=0.7, color=color)
ax.set_title(f'DBSCAN (eps={EPS_M}m)\n→ Q2: 자연 수요 핫스팟 {len(uc)}개 + 저이용 대여소 식별')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')

plt.tight_layout()
plt.show()

### 📋 두 결과의 비교 및 의미

| 기준 | K-Means | DBSCAN |
|---|---|---|
| 비즈니스 용도 | 재배치 트럭 배치 & 구역 책임 | 핫스팟 마케팅 / 저이용 대여소 발굴 |
| 모든 대여소 할당 | 100% (강제) | 핫스팟 or 노이즈 |
| 클러스터 개수 | 경영 결정 (k=트럭 대수) | 데이터가 결정 |
| 저이용 대여소 | 어느 한 트럭에 할당 | 노이즈로 자동 식별 |

**핵심 메시지**: **서로 다른 질문에 서로 다른 답.** 실무에서는 둘 다 동시에 활용하는 경우가 많음!

## 5. Folium 인터랙티브 지도

In [ ]:
m = folium.Map(location=[37.55, 126.98], zoom_start=11, tiles='CartoDB positron')

heat_layer = folium.FeatureGroup(name='이용량 Heatmap').add_to(m)
HeatMap(df[['lat', 'lon', 'monthly_usage']].values.tolist(),
        radius=10, blur=14, min_opacity=0.3).add_to(heat_layer)

truck_layer = folium.FeatureGroup(name='K-Means 트럭 차고지').add_to(m)
for i, (lat, lon) in enumerate(centers_kmeans):
    folium.Marker(
        location=[lat, lon],
        popup=f'<b>Truck Base {i}</b>',
        icon=folium.Icon(color='red', icon='truck', prefix='fa'),
    ).add_to(truck_layer)

df['hotspot'] = labels_dbscan

hotspot_summary = df[df['hotspot'] != -1].groupby('hotspot').agg(
    n_stations=('station_id', 'count'),
    total_usage=('monthly_usage', 'sum'),
    avg_usage=('monthly_usage', 'mean'),
    top_gu=('gu', lambda x: x.mode()[0]),
    center_lat=('lat', 'mean'),
    center_lon=('lon', 'mean'),
).round(1).sort_values('total_usage', ascending=False)

hotspot_layer = folium.FeatureGroup(name='DBSCAN 핫스팟').add_to(m)
for _, row in hotspot_summary.iterrows():
    folium.CircleMarker(
        location=[row['center_lat'], row['center_lon']],
        radius=np.sqrt(row['total_usage']) / 80,
        popup=f"<b>Hotspot ({row['top_gu']})</b><br>"
              f"{int(row['n_stations'])} 대여소<br>"
              f"월 {int(row['total_usage']):,} 건",
        color='blue', fill=True, fillColor='blue', fillOpacity=0.35,
    ).add_to(hotspot_layer)

noise_layer = folium.FeatureGroup(name='저이용 대여소 (노이즈)', show=False).add_to(m)
for _, row in df[df['hotspot'] == -1].iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=2.5,
        popup=f"<b>{row['station_name']}</b><br>"
              f"{row['gu']}<br>월 {int(row['monthly_usage'])}건",
        color='gray', fill=True, fillColor='gray', fillOpacity=0.5,
    ).add_to(noise_layer)

folium.LayerControl(collapsed=False).add_to(m)
m

- 🔴 **빨간 트럭 아이콘**: K-Means 재배치 트럭 차고지
- 🔵 **파란 원**: DBSCAN 핫스팟 (원 크기 = 총 월간 이용량)
- ⚪ **회색 점**: 노이즈(저이용) 대여소 (기본 숨김 — 좌상단에서 켜기)
- 🔥 **배경 Heatmap**: 전체 이용량 밀도

## 6. 추가 과제 및 토론 사항

### 🎯 추가 과제

1. **[하이퍼파라미터 민감도]** `eps`를 500m, 800m, 1200m로, `min_samples`를 5, 10, 20으로 조합(9 케이스)하여 DBSCAN 결과를 비교하라. 각 조합의 **#핫스팟, 노이즈 비율, Top1 크기**를 표로 정리하라.

2. **[HDBSCAN 도전]** `pip install hdbscan` 후 동일 데이터에 HDBSCAN을 적용. 서울 도심(고밀도)과 외곽(저밀도)이 공존하는 상황에서 단일 `eps` 방식보다 어떤 장점이 있는가? `min_cluster_size=20`으로 시작하라.

3. **[수송 문제 최적화]** 각 핫스팟을 수요 노드, 각 트럭 거점을 공급 노드로 보고, `scipy.optimize.linprog`로 최적 재배치 경로를 계산하라. 수요-공급 불균형을 최소화하는 일일 수송량은?

### 💬 토론 질문

1. **관악구·강북구에 노이즈 비율이 높다.** 이를 근거로 해당 지역 대여소를 줄이는 정책을 편다면, 어떤 윤리적/형평성 이슈가 발생하는가? 공공서비스 제공자로서의 책임은?

2. **본 실습의 evaluation은 전적으로 시각적/정성적이었다.** 정답 레이블이 없는 상황에서 "좋은 군집화"를 정의하는 방법들(Silhouette, Davies-Bouldin, Gap statistic 등)의 **장단점과 도메인 의존성**을 논하라.

3. **이 문제를 실시간(streaming)으로 확장**한다면? 매 5분마다 실시간 대여 현황이 들어올 때, 매번 전체 DBSCAN을 재실행하는 것은 비효율적이다. 왜 그런가?